# Estimator Sheet 
This notebook builds the single combined dataset used for quantile regression estimation. It walks a folder of forecast-card CSVs (`project-*.csv`, `poi-*.csv`, `observations-*.csv`, and optional `forecast-*.csv` files), finds every project, combines each project's files into one merged table, and writes everything out to a single `combined_output.csv`. That combined file is the input for the model-fitting notebook.

**How to use this notebook:**
1. Run the "find project folders" cell to confirm every project was located. Set `DATA_ROOT` to the folder containing all your project CSVs.
2. Run the "build the combined dataset" cell. For each project this loads its project, poi, observations, and forecast CSVs, merges them together, and labels each observation as `before` or `after` the project's actual open date. All projects are then stacked into one `master_df`.
3. Run the "spot-check" cell to visually confirm the merge worked as expected.
4. Run the "write out" cell to save `master_df` to `combined_output.csv`.

In [ ]:
import pandas as pd
import numpy as np
import glob
import os
import re

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

## 1. Find your project folders

Point `DATA_ROOT` at the folder holding your project data. This walks the entire directory tree underneath it and collects every sub-folder that contains a `project-*.csv` file (each matching folder is treated as one project). The result is a sorted list of folder paths, printed below so you can confirm the right projects were found.

**Update this path to point at your own data folder before continuing.**

In [ ]:
def find_project_folders(root_dir):
    """Return every sub-folder of root_dir that contains a project-*.csv file."""
    folders = []
    for dirpath, _dirnames, filenames in os.walk(root_dir):
        if any(re.match(r"project-.*\.csv$", f, re.IGNORECASE) for f in filenames):
            folders.append(dirpath)
    return sorted(folders)

DATA_ROOT = r"c:\Users\hmod225\Documents\Code\forecastcards_ba_data\data"
project_folders = find_project_folders(DATA_ROOT)
project_folders

## 2. Build the combined dataset

This section does the actual finding, loading, and merging for every project, then stacks the results into one master table used for estimation.

**Loading a project's files** — for a given project folder, finds and reads:
- `project-*.csv` → project-level metadata (one row per project)
- `poi-*.csv` → "points of interest" tied to that project (e.g. intersections/segments)
- `observations-*.csv` → observed traffic/volume records
- `forecast-*.csv` → **optional** — loaded if present, otherwise treated as missing

**Merging a project's files** — performs three sequential merges:
- **project → poi:** tags each POI row with its `project_id`, then merges project metadata onto every POI row (keeping all POIs, broadcasting the single project row across them).
- **before/after labeling:** converts `observation_date` to a real datetime and compares it to the project's `date_open_actual` to create `before_or_after`.
- **(project+poi) → observations:** merges the project/poi table onto the observations, matching on `poi_id` **and** `before_or_after`, keeping every observation row.
- **+ forecast:** if a forecast file exists, merges it in on `poi_id`, suffixing overlapping columns `_observation`/`_forecast`; otherwise renames `start_time`/`end_time` to the `_observation` versions so column naming stays consistent either way.

In [ ]:
def load_project_files(folder):
    """Load the project / poi / observations / (optional) forecast csvs for one folder."""
    project_path = glob.glob(os.path.join(folder, "project-*.csv"))[0]
    poi_path = glob.glob(os.path.join(folder, "poi-*.csv"))[0]
    obs_path = glob.glob(os.path.join(folder, "observations-*.csv"))[0]
    forecast_matches = glob.glob(os.path.join(folder, "forecast-*.csv"))

    project_df = pd.read_csv(project_path)
    poi_df = pd.read_csv(poi_path)
    obs_df = pd.read_csv(obs_path)
    forecast_df = pd.read_csv(forecast_matches[0]) if forecast_matches else None

    return project_df, poi_df, obs_df, forecast_df

In [ ]:
def process_project_folder(folder):
    project_df, poi_df, obs_df, forecast_df = load_project_files(folder)

    project_id = project_df["project_id"].iloc[0]
    date_open_actual = pd.to_datetime(project_df["date_open_actual"].iloc[0])

    poi_df = poi_df.copy()
    poi_df["project_id"] = project_id

    # STEP 1: project -> poi
    # project_df is the left frame, so its columns land first in the result.
    # how="right" keeps every poi row, broadcasting the single project row onto each.
    project_poi = pd.merge(project_df, poi_df, on="project_id", how="right")

    # derive a sensible before/after label for each observation
    # an observation counts as "before" if it happened before the project's
    # actual open date, and "after" otherwise
    obs_df = obs_df.copy()
    obs_df["observation_date"] = pd.to_datetime(obs_df["observation_date"], errors="coerce")
    obs_df["before_or_after"] = np.where(
        obs_df["observation_date"] < date_open_actual, "before", "after"
    )
    # binary flag: before -> 0, after -> 1
    obs_df["before_or_after_binary"] = (obs_df["before_or_after"] == "after").astype(int)

    # STEP 2: (project + poi) -> observation
    # project_poi is again the LEFT frame, so project/poi columns stay ahead of the
    # observation columns that get appended on the right. how="right" keeps every
    # observation row, attaching the poi snapshot that matches its before/after label.
    merged = pd.merge(
        project_poi, obs_df, on=["poi_id", "before_or_after"], how="right"
    )

    # STEP 3: + forecast (broadcast across every obs of that POI)
    if forecast_df is not None:
        merged = pd.merge(
            merged, forecast_df, on="poi_id", how="left",
            suffixes=("_observation", "_forecast")
        )
    else:
        # rename raw obs-only columns to match the suffixed naming used when a
        # forecast file IS present, so column names stay consistent across projects
        merged = merged.rename(columns={"start_time": "start_time_observation",
                                         "end_time": "end_time_observation"})

    return merged

In [ ]:

project_tables = [process_project_folder(folder) for folder in project_folders]

master_df = pd.concat(project_tables, ignore_index=True, sort=False)

print(f"{len(master_df)} total observation rows across {len(project_tables)} project(s)")
master_df.head(10)

## 3. Spot-check the merged data

Displays a narrowed view of just the columns most relevant for spot-checking, the join keys, dates, before/after labels, and a few geometric attributes (lane counts, shoulder width), to visually verify the merges worked correctly before writing anything out.

In [ ]:
master_df[[
    "project_id", "poi_id", "observation_date", "date_open_actual",
    "before_or_after", "before_or_after_binary",
    "through_lanes", "right_turn_lanes", "shoulder_width"
]]

## 4. Write out the combined CSV

Saves `master_df` to `combined_output.csv` in the working directory and confirms how many rows/columns were written. This file is the input for the quantile regression model-fitting notebook.

In [ ]:
output_path = "combined_output.csv"
master_df.to_csv(output_path, index=False)
print(f"Wrote {len(master_df)} rows and {len(master_df.columns)} columns to {output_path}")